# SentimentClassifier
Textklassifikation & Sentiment (nicht nur Tokenizing!)

In [4]:
#!pip install emoji

# Ablauf der Funktionskette

    model(input_vector)
        │
        ▼
    SentimentClassifier.__call__(input_vector)   (geerbt von nn.Module)
        │
        ▼
    SentimentClassifier.forward(input_vector)
        │
        ▼
    self.fc(text_vector)
        │
        ▼
    Linear.__call__(text_vector)
        │
        ▼
    Linear.forward(text_vector)

    model(input_vector)
    
     → __call__()
     → forward()
     → fc(...)
     → Linear.__call__()
     → Linear.forward()
     
## class Linear(Module):
     class Linear(Module):

    def forward(self, input):
        return torch.nn.functional.linear(
            input,
            self.weight,
            self.bias
        )
        output = input @ weight.T + bias

## nn.Linear erbt von nn.Module
    Damit bekommt Linear automatisch:

    __call__()

    parameters()

    train()

    eval()

    state_dict()

    register_buffer()

    register_parameter()

## das Modell noch nicht trainiert. Ergebnis macht keinen Sinn!!

In [10]:
import spacy
import torch
import torch.nn as nn
from spacy.tokens import Doc

# 1. spaCy für die Vorverarbeitung (Tokenizing & Vokabular)
nlp = spacy.load("de_core_news_sm")

# 2. Ein minimales PyTorch-Modell (Sentiment: Positiv oder Negativ)
class SentimentClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.fc = nn.Linear(embedding_dim, 2) # 2 Ausgabeklassen
        
    def forward(self, text_vector):
        return self.fc(text_vector)

# Modell initialisieren (96 ist die Standardgröße der spaCy-Vektoren in 'sm')
model = SentimentClassifier(embedding_dim=96, hidden_dim=32) # hidden_dim nicht benutzt

# 3. Text mit spaCy verarbeiten
text = "Dieser Kurs ist absolut fantastisch!"
doc = nlp(text)

# spaCy übernimmt das 'Batching' & Vektorisieren
# Wir nutzen den Durchschnittsvektor des gesamten Satzes (doc.vector)
#print(doc.vector)
input_vector = torch.tensor(doc.vector).unsqueeze(0) # Batch-Dimension hinzufügen
print(input_vector.shape)
print(input_vector)
# 4. Vorhersage mit PyTorch
model.eval()
with torch.no_grad():
    output = model(input_vector)
    print("output", output)
    prediction = torch.softmax(output, dim=1)
    print("prediction", prediction)

print(f"Text: {text}")
print(f"Wahrscheinlichkeiten (Negativ, Positiv): {prediction.numpy()}")


torch.Size([1, 96])
tensor([[ 1.6802e-01,  6.2245e-01,  2.1178e+00,  2.1515e-01, -3.3498e-01,
         -4.2460e-01,  1.2009e+00,  1.0567e+00, -2.6151e-01, -9.0191e-01,
          1.0704e+00, -1.3359e+00,  4.9581e-01,  6.4083e-01,  4.8793e-01,
          1.6002e+00,  6.4645e-02,  1.2747e+00,  6.3597e-01, -1.4699e-01,
          6.9257e-01,  2.5375e+00,  1.9122e+00,  1.1654e+00, -7.3040e-01,
         -1.6673e+00,  2.6912e+00, -1.8231e+00,  1.8630e-01, -1.0653e+00,
          5.0170e-01, -1.0446e+00, -3.7409e-01, -1.1571e+00, -2.4945e-01,
         -1.2913e-01, -1.4200e+00, -1.3180e+00,  3.6364e-01,  9.6380e-01,
          8.5086e-01,  1.7182e+00, -6.4694e-01,  6.9647e-01,  6.8649e-01,
          9.8940e-01, -9.2478e-01, -3.0293e-01,  8.0135e-01, -8.9183e-02,
         -2.5367e-01, -3.1955e-01, -8.5097e-01,  4.2154e-01, -1.4669e-03,
         -9.2707e-01, -8.1368e-01,  6.4016e-01, -5.7082e-01, -9.0618e-01,
         -9.9867e-01,  6.4129e-01,  4.5262e-02, -3.5888e-01, -1.6919e+00,
         -2.0294e-

In [ ]:
## 

In [5]:
import spacy
import torch
import torch.nn as nn

# 1. Vorbereitung
nlp = spacy.load("de_core_news_sm")
torch.manual_seed(42)

# 2. Modell-Definition
class SentimentClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.layer1 = nn.Linear(embedding_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

model = SentimentClassifier(96, 32)

# 3. 50 Trainingssätze erstellen
pos_texts = [
    "Das ist super!", "Ich liebe es.", "Einfach fantastisch.", "Sehr gut gemacht.", "Ich bin begeistert.",
    "Wunderbare Arbeit.", "Klasse Leistung.", "Absolut empfehlenswert.", "Ein echtes Highlight.", "Top Qualität.",
    "Sehr hilfreich.", "Ich bin sehr zufrieden.", "Großartig!", "Beste Entscheidung.", "Es macht Spaß.",
    "Perfekt gelaufen.", "Toller Service.", "Sehr freundlich.", "Beeindruckend.", "Gerne wieder.",
    "Alles bestens.", "Einwandfrei.", "Hervorragend.", "Spitzenklasse.", "Einfach nur toll."
]

neg_texts = [
    "Das ist schrecklich.", "Ich hasse es.", "Ganz furchtbar.", "Sehr schlecht.", "Ich bin enttäuscht.",
    "Miese Qualität.", "Nicht zu gebrauchen.", "Verschwendung von Zeit.", "Ein totaler Reinfall.", "Unterirdisch.",
    "Überhaupt nicht hilfreich.", "Ich bin unzufrieden.", "Grauenhaft!", "Fehlkauf.", "Es macht keinen Sinn.",
    "Viel zu teuer.", "Schlechter Service.", "Sehr unfreundlich.", "Enttäuschend.", "Nie wieder.",
    "Alles kaputt.", "Mangelhaft.", "Ungenügend.", "Katastrophe.", "Einfach nur mies."
]

# Texte und Labels (1 = Positiv, 0 = Negativ) kombinieren
train_texts = pos_texts + neg_texts
labels = torch.tensor([1]*25 + [0]*25) 
print("labels:", labels)
# Texte in spaCy-Vektoren umwandeln
train_vectors = torch.stack([torch.tensor(nlp(t).vector) for t in train_texts])

# 4. Training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(150): # Mehr Daten brauchen manchmal mehr Epochen
    optimizer.zero_grad()
    outputs = model(train_vectors)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 5. Test
model.eval()
test_text = "Die Qualität ist wirklich spitze!"
test_vec = torch.tensor(nlp(test_text).vector).unsqueeze(0)

with torch.no_grad():
    res = torch.softmax(model(test_vec), dim=1)

print(f"\nTest: {test_text}")
print(f"Wahrscheinlichkeit (Negativ, Positiv): {res.numpy()}")


labels: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0])
Epoch 50, Loss: 0.0012
Epoch 100, Loss: 0.0004
Epoch 150, Loss: 0.0003

Test: Die Qualität ist wirklich spitze!
Wahrscheinlichkeit (Negativ, Positiv): [[0.09082806 0.909172  ]]


## Besipiel mit mehr Testtexte

In [6]:
import spacy
import torch
import torch.nn as nn

# 1. Vorbereitung
nlp = spacy.load("de_core_news_sm")
torch.manual_seed(42)

# 2. Modell-Definition
class SentimentClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super().__init__()
        self.layer1 = nn.Linear(embedding_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

model = SentimentClassifier(96, 32)

# 3. Trainingsdaten
pos_texts = [
    "Das ist super!", "Ich liebe es.", "Einfach fantastisch.", "Sehr gut gemacht.", "Ich bin begeistert.",
    "Wunderbare Arbeit.", "Klasse Leistung.", "Absolut empfehlenswert.", "Ein echtes Highlight.", "Top Qualität.",
    "Sehr hilfreich.", "Ich bin sehr zufrieden.", "Großartig!", "Beste Entscheidung.", "Es macht Spaß.",
    "Perfekt gelaufen.", "Toller Service.", "Sehr freundlich.", "Beeindruckend.", "Gerne wieder.",
    "Alles bestens.", "Einwandfrei.", "Hervorragend.", "Spitzenklasse.", "Einfach nur toll."
]

neg_texts = [
    "Das ist schrecklich.", "Ich hasse es.", "Ganz furchtbar.", "Sehr schlecht.", "Ich bin enttäuscht.",
    "Miese Qualität.", "Nicht zu gebrauchen.", "Verschwendung von Zeit.", "Ein totaler Reinfall.", "Unterirdisch.",
    "Überhaupt nicht hilfreich.", "Ich bin unzufrieden.", "Grauenhaft!", "Fehlkauf.", "Es macht keinen Sinn.",
    "Viel zu teuer.", "Schlechter Service.", "Sehr unfreundlich.", "Enttäuschend.", "Nie wieder.",
    "Alles kaputt.", "Mangelhaft.", "Ungenügend.", "Katastrophe.", "Einfach nur mies."
]

train_texts = pos_texts + neg_texts
labels = torch.tensor([1]*25 + [0]*25)

train_vectors = torch.stack([torch.tensor(nlp(t).vector) for t in train_texts])

# 4. Training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(150):
    optimizer.zero_grad()
    outputs = model(train_vectors)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 5. Mehrere Test-Sätze
tests = [
    "Der Kurs ist nicht schlecht.",
    "Der Service ist okay.",
    "Das Produkt ist teuer.",
    "Ich bin überrascht wie gut das ist.",
    "Der Film war langweilig aber schön gefilmt."
]

model.eval()
with torch.no_grad():
    for test_text in tests:
        test_vec = torch.tensor(nlp(test_text).vector).unsqueeze(0)
        output = model(test_vec)
        probs = torch.softmax(output, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        label_str = "Positiv" if predicted_class == 1 else "Negativ"
        print(f"\nText: {test_text}")
        print(f"Wahrscheinlichkeiten (Negativ, Positiv): {probs.numpy()}")
        print(f"Predicted Class: {label_str}")

Epoch 50, Loss: 0.0012
Epoch 100, Loss: 0.0004
Epoch 150, Loss: 0.0003

Text: Der Kurs ist nicht schlecht.
Wahrscheinlichkeiten (Negativ, Positiv): [[9.9949956e-01 5.0048664e-04]]
Predicted Class: Negativ

Text: Der Service ist okay.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.1728792  0.82712084]]
Predicted Class: Positiv

Text: Das Produkt ist teuer.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.9758724  0.02412757]]
Predicted Class: Negativ

Text: Ich bin überrascht wie gut das ist.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.02575739 0.9742426 ]]
Predicted Class: Positiv

Text: Der Film war langweilig aber schön gefilmt.
Wahrscheinlichkeiten (Negativ, Positiv): [[0.02922294 0.9707771 ]]
Predicted Class: Positiv
